In [ ]:
from pathlib import Path

import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader

import fluidbench
from fluidbench.cfdataset import CFDataset

In [ ]:
ID = "cylinder_Re41_interp"
dataset_path = fluidbench.download(ID)

In [ ]:
def load_split(split):
    folder = dataset_path / split
    cases = sorted((path for path in folder.glob("*.h5") if path.stem != "constants"), key=lambda path: int(path.stem))
    data = []
    for case in cases:
        with h5py.File(case, "r") as file:
            data.append((file["scalars"][:], file["fields"][:]))
    return data

def to_dataset(cases):
    # Each snapshot is paired with the next one: inputs are snapshot i, targets snapshot i + 1.
    scalars = np.concatenate([scalars[:-1] for scalars, fields in cases])
    inputs = np.concatenate([fields[:-1] for scalars, fields in cases])
    targets = np.concatenate([fields[1:] for scalars, fields in cases])
    return CFDataset(scalars, inputs, targets, np.arange(len(targets)))

In [ ]:
# The benchmark ships train and test only; hold out the end of each training case for validation.
VAL_FRACTION = 0.1
train_cases, validation_cases = [], []
for scalars, fields in load_split("train"):
    cut = int(len(fields) * (1 - VAL_FRACTION))
    train_cases.append((scalars[:cut], fields[:cut]))
    validation_cases.append((scalars[cut:], fields[cut:]))

train_dataset = to_dataset(train_cases)
validation_dataset = to_dataset(validation_cases)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=8, shuffle=False)
len(train_dataset), len(validation_dataset)

In [ ]:
### Model definition
from minimal_models import BaselineZero as architecture

weights_path = Path("weights") / ID / f"{architecture.__name__}.pt"

model = architecture()
model.to(model.device)

In [ ]:
history = model.fit(train_loader, validation_loader)

In [ ]:
weights_path.parent.mkdir(parents=True, exist_ok=True)
model.save_weights(weights_path)
weights_path